# Nextbus Feeder (Fabric Notebook)

Runs the Nextbus feeder inside Microsoft Fabric and emits CloudEvents to the bound Fabric Event Stream custom endpoint.

**Runtime model**
- The `nextbus` package is installed from Lakehouse-hosted wheels before the feeder starts.
- The Event Stream connection string is looked up at runtime via the Fabric Topology API.
- The notebook runs `nextbus feed` in continuous mode for up to 55 minutes, then exits cleanly so the scheduler can restart it.
- Source docs: ../README.md


In [ ]:
# === PARAMETERS (overwritten by deploy-feeder-notebook.ps1) ===
EVENTSTREAM_NAME = "nextbus-ingest"
STATE_FILE       = "/lakehouse/default/Files/feeder-state/nextbus/state.json"
POLLING_INTERVAL = 30
RUN_DURATION     = 3300
ONCE_MODE        = False
WORKSPACE_ID     = ""


In [ ]:
import datetime, glob as _glob, importlib, pathlib, subprocess, sys

LOG_PATH = '/lakehouse/default/Files/feeder-state/nextbus/last-run.log'
pathlib.Path(LOG_PATH).parent.mkdir(parents=True, exist_ok=True)

def _log(msg):
    line = f'[{datetime.datetime.utcnow().isoformat()}Z] {msg}'
    print(line)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

with open(LOG_PATH, 'w', encoding='utf-8') as f:
    f.write('')

_log('Notebook kernel started; Lakehouse mounted')
_log(f'Python: {sys.version}')

try:
    _wheels = _glob.glob('/lakehouse/default/Files/wheels/nextbus/*.whl')
    if _wheels:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps'] + _wheels
        )
    _pypi_deps = ['avro>=1.11.3', 'dataclasses_json>=0.6.7', 'confluent_kafka>=2.5.3', 'cloudevents>=1.11.0,<2.0.0']
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + _pypi_deps
    )
    for _key in list(sys.modules.keys()):
        if _key.startswith('cloudevents'):
            del sys.modules[_key]
    importlib.invalidate_caches()
    _log(f'Installed {len(_wheels)} wheel(s) + {len(_pypi_deps)} PyPI deps')
except Exception as exc:
    _log(f'pip install/setup failed: {exc}')
    raise


In [ ]:
import json, os, requests

FABRIC_API = 'https://api.fabric.microsoft.com/v1'

try:
    import notebookutils
    token = notebookutils.credentials.getToken('pbi')
except Exception:
    from notebookutils import mssparkutils
    token = mssparkutils.credentials.getToken('pbi')

headers = {'Authorization': f'Bearer {token}'}
workspace_id = WORKSPACE_ID
if not workspace_id:
    try:
        import notebookutils
        workspace_id = notebookutils.runtime.context['currentWorkspaceId']
    except Exception:
        from notebookutils import mssparkutils
        workspace_id = json.loads(mssparkutils.runtime.context)['currentWorkspaceId']

es_list = requests.get(f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams', headers=headers, timeout=30)
es_list.raise_for_status()
es = next((e for e in es_list.json().get('value', []) if e.get('displayName') == EVENTSTREAM_NAME), None)
if not es:
    raise RuntimeError(f"Event Stream {EVENTSTREAM_NAME!r} not found in workspace {workspace_id}.")

es_id = es["id"]
topo = requests.get(f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams/{es_id}/topology', headers=headers, timeout=30)
topo.raise_for_status()
src = next((s for s in topo.json().get('sources', []) if s.get('type') == 'CustomEndpoint'), None)
if not src:
    raise RuntimeError('Event Stream has no CustomEndpoint source.')

src_id = src["id"]
conn = requests.get(
    f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams/{es_id}/sources/{src_id}/connection',
    headers=headers,
    timeout=30,
)
conn.raise_for_status()
os.environ['CONNECTION_STRING'] = conn.json()['accessKeys']['primaryConnectionString']
_log(f'Connection string ready for workspace {workspace_id} / event stream {EVENTSTREAM_NAME}')


In [ ]:
import os, pathlib, sys, threading, traceback
from urllib.parse import parse_qsl

AGENCY = 'sf-muni'
ROUTE = '*'

def _event_hub_name_from_connection_string(connection_string: str) -> str:
    kv = dict(parse_qsl(connection_string.replace(';', '&')))
    return kv.get('EntityPath', '')

try:
    _log('Starting feeder (continuous mode)')
    pathlib.Path(STATE_FILE).parent.mkdir(parents=True, exist_ok=True)
    os.environ['STATE_FILE'] = STATE_FILE
    os.environ['POLLING_INTERVAL'] = str(POLLING_INTERVAL)
    os.environ['ONCE_MODE'] = 'true' if ONCE_MODE else 'false'
    os.environ['AGENCY'] = AGENCY
    os.environ['ROUTE'] = ROUTE
    cs = os.environ['CONNECTION_STRING']
    event_hub_name = _event_hub_name_from_connection_string(cs)
    if not event_hub_name:
        raise RuntimeError('Event Stream connection string does not include EntityPath.')
    os.environ['FEED_CONNECTION_STRING'] = cs
    os.environ['FEED_EVENT_HUB_NAME'] = event_hub_name
    os.environ['REFERENCE_CONNECTION_STRING'] = cs
    os.environ['REFERENCE_EVENT_HUB_NAME'] = event_hub_name
    _log(f'CONNECTION_STRING present: {bool(cs)}')
    _log(f'Using agency={AGENCY}, route={ROUTE}, event hub={event_hub_name}')

    _log('Importing nextbus package from Fabric Environment...')
    from nextbus import nextbus as feeder
    _log(f'Imported feeder from: {feeder.__file__}')

    argv_backup = sys.argv
    try:
        sys.argv = ['nextbus', 'feed', '--once'] if ONCE_MODE else ['nextbus', 'feed']
        _log(f'Running feeder.main() with argv={sys.argv}, duration={RUN_DURATION}s')
        _err = []
        def _worker():
            try:
                feeder.main()
            except BaseException as e:
                _err.append(e)
        t = threading.Thread(target=_worker, daemon=True)
        t.start()
        t.join(timeout=None if ONCE_MODE else RUN_DURATION)
        if t.is_alive():
            _log(f'Run duration {RUN_DURATION}s reached; exiting cleanly.')
        elif _err:
            raise _err[0]
    finally:
        sys.argv = argv_backup
    _log('Cycle complete.')
    try:
        import notebookutils
        notebookutils.notebook.exit('OK')
    except Exception:
        pass
except Exception as exc:
    tb = traceback.format_exc()
    _log(f'FAILED: {exc}\n{tb}')
    try:
        import notebookutils
        notebookutils.notebook.exit(f'FAIL: {exc}')
    except Exception:
        pass
    raise
